In [21]:
import pandas as pd

In [22]:
power = pd.read_csv('/Users/vedikashirtekar/MEDS/EDS-213/eds-213-labs/data/DATA/power/electricity-generation_emissions_sources_v5_5_0.csv')
non_res = pd.read_csv('/Users/vedikashirtekar/MEDS/EDS-213/eds-213-labs/data/DATA/buildings/non-residential-onsite-fuel-usage_emissions_sources_v5_5_0.csv')
#power = pd.read_csv('/Users/vedikashirtekar/MEDS/EDS-213/eds-213-labs/data/DATA/power/electricity-generation_emissions_sources_v5_5_0.csv')


In [23]:
# Explore the data structures
print(non_res.shape, power.shape)
print(non_res.dtypes)

(207888, 43) (154879, 43)
source_id                   int64
source_name                object
source_type               float64
iso3_country               object
sector                     object
subsector                  object
start_time                 object
end_time                   object
lat                       float64
lon                       float64
geometry_ref               object
gas                        object
emissions_quantity        float64
temporal_granularity       object
activity                  float64
activity_units             object
emissions_factor          float64
emissions_factor_units     object
capacity                  float64
capacity_units             object
capacity_factor           float64
other1                     object
other1_def                 object
other2                     object
other2_def                 object
other3                     object
other3_def                 object
other4                    float64
other4_def            

In [ ]:
# Drop always null or useless columns
# Some columns (source_type, geometry_ref, sector, and subsector) is entirely null, not relevant, are has a constant value
drop_cols = ['source_type', 'geometry_ref', 'sector', 'subsector', 'created_date', 'modified_date']
non_res = non_res.drop(columns=drop_cols, errors='ignore')
power = power.drop(columns=drop_cols, errors='ignore')

In [25]:
# Parse datetime columns
non_res['start_time'] = pd.to_datetime(non_res['start_time'])
non_res['end_time'] = pd.to_datetime(non_res['end_time'])
# same for power
power['start_time'] = pd.to_datetime(power['start_time'])
power['end_time'] = pd.to_datetime(power['end_time'])

In [26]:
# Are there any duplicates? 
# Valid dataset should have one row per source_id + start_time + gas combination
dupes = non_res.duplicated(subset=['source_id', 'start_time', 'gas'])
print(dupes.sum())

0


In [27]:
# Check for nulls and outliers in numeric fields
numeric_cols = ['emissions_quantity', 'activity', 'emissions_factor', 'capacity', 'lat', 'lon']
print(non_res[numeric_cols].describe())
print(non_res[numeric_cols].isnull().sum())

       emissions_quantity      activity  emissions_factor      capacity  \
count       207888.000000  2.078880e+05      2.078880e+05  2.078880e+05   
mean          5293.175706  8.615382e+07      7.349123e-05  2.670397e+06   
std          24256.979637  3.883335e+08      8.532392e-05  1.141074e+07   
min              0.000000  0.000000e+00      9.587420e-07  0.000000e+00   
25%            152.193098  2.354243e+06      5.001449e-05  7.977713e+04   
50%            593.892610  9.499643e+06      6.296034e-05  3.128240e+05   
75%           2641.845250  4.269371e+07      8.130056e-05  1.381689e+06   
max         981715.594000  1.618478e+10      2.437498e-03  3.365205e+08   

                 lat            lon  
count  192028.000000  192028.000000  
mean       38.454181     -92.203442  
std         5.276979      12.787112  
min        19.596185    -164.442490  
25%        34.698653     -98.207827  
50%        38.399181     -90.357245  
75%        41.851047     -83.431854  
max        69.356849

In [28]:
# Same for power
#numeric_cols = ['emissions_quantity', 'activity', 'emissions_factor', 'capacity', 'lat', 'lon']
print(power[numeric_cols].describe())
print(power[numeric_cols].isnull().sum())

       emissions_quantity      activity  emissions_factor       capacity  \
count        1.548790e+05  1.548790e+05     154879.000000  154879.000000   
mean         4.834783e+04  9.023929e+04          0.426030     318.851836   
std          1.256916e+05  1.852249e+05          0.310532     519.207310   
min          0.000000e+00  0.000000e+00          0.000000       0.000000   
25%          4.760500e+02  1.803000e+03          0.320950      15.900000   
50%          4.402000e+03  1.222000e+04          0.449360      70.000000   
75%          3.605000e+04  7.439500e+04          0.513628     432.000000   
max          1.958000e+06  2.300000e+06          1.473934    4329.600000   

                 lat            lon  
count  154879.000000  154879.000000  
mean       38.097927     -93.079859  
std         5.928407      17.128114  
min        19.631600    -166.553200  
25%        33.869700     -98.322800  
50%        38.985800     -89.589100  
75%        41.663100     -81.059200  
max        

In [32]:
# Drop "other" columns in non_res and power
other_cols = [c for c in non_res.columns if c.startswith('other')]
non_res = non_res.drop(columns=other_cols)
#power = power.drop(columns=other_cols)

In [33]:
# Drop "other" columns in power
other_cols = [c for c in power.columns if c.startswith('other')]
power = power.drop(columns=other_cols)

In [36]:
# Create sources table
sources = non_res[['source_id', 'source_name', 'iso3_country', 'lat', 'lon', 
                    'capacity', 'capacity_units', 'capacity_factor']].drop_duplicates(subset='source_id')
sources_power = power[['source_id', 'source_name', 'iso3_country', 'lat', 'lon',
                        'capacity', 'capacity_units', 'capacity_factor']].drop_duplicates(subset='source_id')

sources = pd.concat([sources, sources_power]).drop_duplicates(subset='source_id').reset_index(drop=True)

In [37]:
record_cols = ['source_id', 'start_time', 'end_time', 'gas', 'emissions_quantity',
               'temporal_granularity', 'activity', 'activity_units', 
               'emissions_factor', 'emissions_factor_units']

non_records = non_res[record_cols].copy()
non_records['sector'] = 'buildings'

power_records = power[record_cols].copy()
power_records['sector'] = 'power'

emission_records = pd.concat([non_records, power_records]).reset_index(drop=True)

In [39]:
# Export
sources.to_csv('sources.csv', index=False)
emission_records.to_csv('emission_records.csv', index=False)

In [44]:
emission_records.dtypes

source_id                          int64
start_time                datetime64[ns]
end_time                  datetime64[ns]
gas                               object
emissions_quantity               float64
temporal_granularity              object
activity                         float64
activity_units                    object
emissions_factor                 float64
emissions_factor_units            object
sector                            object
dtype: object